<a href="https://colab.research.google.com/github/aviralraghav/data-analyst-assessment-olist/blob/main/Virtubox_DataAnalyst_assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np

customers = pd.read_csv("olist_customers_dataset.csv")
geolocation = pd.read_csv("olist_geolocation_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")
reviews = pd.read_csv("olist_order_reviews_dataset.csv")
orders = pd.read_csv("olist_orders_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
sellers = pd.read_csv("olist_sellers_dataset.csv")
translation = pd.read_csv("product_category_name_translation.csv")

print("All 9 datasets loaded successfully!")

All 9 datasets loaded successfully!


In [4]:
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "translation": translation
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

customers: 99,441 rows × 5 columns
geolocation: 1,000,163 rows × 5 columns
order_items: 112,650 rows × 7 columns
payments: 103,886 rows × 5 columns
reviews: 99,224 rows × 7 columns
orders: 99,441 rows × 8 columns
products: 32,951 rows × 9 columns
sellers: 3,095 rows × 4 columns
translation: 71 rows × 2 columns


In [5]:
for name, df in datasets.items():
    print(f"\n===== {name.upper()} =====")
    print("Duplicate rows:", df.duplicated().sum())
    print("Missing values:")
    print(df.isna().sum()[df.isna().sum() > 0])


===== CUSTOMERS =====
Duplicate rows: 0
Missing values:
Series([], dtype: int64)

===== GEOLOCATION =====
Duplicate rows: 261831
Missing values:
Series([], dtype: int64)

===== ORDER_ITEMS =====
Duplicate rows: 0
Missing values:
Series([], dtype: int64)

===== PAYMENTS =====
Duplicate rows: 0
Missing values:
Series([], dtype: int64)

===== REVIEWS =====
Duplicate rows: 0
Missing values:
review_comment_title      87656
review_comment_message    58247
dtype: int64

===== ORDERS =====
Duplicate rows: 0
Missing values:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

===== PRODUCTS =====
Duplicate rows: 0
Missing values:
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

===== SEL

In [6]:
# Q3 – Data cleaning and preparation

# 1. Remove exact duplicate rows
for name, df in datasets.items():
    datasets[name] = df.drop_duplicates().copy()

# 2. Convert date columns
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    if col in orders.columns:
        orders[col] = pd.to_datetime(orders[col], errors="coerce")

# 3. Create analysis-ready transaction dataset
analysis_df = (
    order_items
    .merge(orders, on="order_id", how="left")
    .merge(products, on="product_id", how="left")
    .merge(sellers, on="seller_id", how="left")
    .merge(translation, on="product_category_name", how="left")
)

# 4. Aggregate payments at order level
payment_summary = (
    payments.groupby("order_id", as_index=False)
    .agg(
        total_payment_value=("payment_value", "sum"),
        max_installments=("payment_installments", "max")
    )
)

analysis_df = analysis_df.merge(payment_summary, on="order_id", how="left")

# 5. Aggregate reviews at order level
review_summary = (
    reviews.groupby("order_id", as_index=False)
    .agg(
        review_score=("review_score", "mean"),
        review_count=("review_id", "count")
    )
)

analysis_df = analysis_df.merge(review_summary, on="order_id", how="left")

# 6. Calculated fields
analysis_df["item_revenue"] = (
    analysis_df["price"].fillna(0) +
    analysis_df["freight_value"].fillna(0)
)

analysis_df["delivery_days"] = (
    analysis_df["order_delivered_customer_date"]
    - analysis_df["order_purchase_timestamp"]
).dt.days

analysis_df["delivery_delay_days"] = (
    analysis_df["order_delivered_customer_date"]
    - analysis_df["order_estimated_delivery_date"]
).dt.days

analysis_df["delivery_status"] = np.where(
    analysis_df["delivery_delay_days"] > 0,
    "Late",
    "On Time"
)

# 7. Standardize missing category values
if "product_category_name_english" in analysis_df.columns:
    analysis_df["product_category_name_english"] = (
        analysis_df["product_category_name_english"].fillna("Unknown")
    )

print("Processed dataset shape:", analysis_df.shape)
print("Processing completed successfully!")

Processed dataset shape: (112650, 34)
Processing completed successfully!


In [7]:
# Save processed data
analysis_df.to_csv("processed_olist_analysis.csv", index=False)

print("Saved: processed_olist_analysis.csv")

Saved: processed_olist_analysis.csv


In [9]:
import pandas as pd
import numpy as np
import os

base_path = "/content"

files = {}

for file in os.listdir(base_path):
    if file.endswith(".csv"):
        files[file] = pd.read_csv(os.path.join(base_path, file))

print("Files loaded:")
for name, df in files.items():
    print(f"{name}: {df.shape}")

Files loaded:
olist_order_items_dataset.csv: (112650, 7)
olist_products_dataset.csv: (32951, 9)
product_category_name_translation.csv: (71, 2)
olist_sellers_dataset.csv: (3095, 4)
olist_order_reviews_dataset.csv: (99224, 7)
olist_customers_dataset.csv: (99441, 5)
processed_olist_analysis.csv: (112650, 34)
olist_orders_dataset.csv: (99441, 8)
olist_geolocation_dataset.csv: (1000163, 5)
olist_order_payments_dataset.csv: (103886, 5)


In [10]:
# Q3: Data Quality Audit

raw_files = {k: v for k, v in files.items()
             if k != "processed_olist_analysis.csv"}

for name, df in raw_files.items():
    print("\n" + "="*60)
    print(name)
    print("Shape:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())
    print("Missing values:")
    print(df.isna().sum()[df.isna().sum() > 0])


olist_order_items_dataset.csv
Shape: (112650, 7)
Duplicate rows: 0
Missing values:
Series([], dtype: int64)

olist_products_dataset.csv
Shape: (32951, 9)
Duplicate rows: 0
Missing values:
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

product_category_name_translation.csv
Shape: (71, 2)
Duplicate rows: 0
Missing values:
Series([], dtype: int64)

olist_sellers_dataset.csv
Shape: (3095, 4)
Duplicate rows: 0
Missing values:
Series([], dtype: int64)

olist_order_reviews_dataset.csv
Shape: (99224, 7)
Duplicate rows: 0
Missing values:
review_comment_title      87656
review_comment_message    58247
dtype: int64

olist_customers_dataset.csv
Shape: (99441, 5)
Duplicate rows: 0
Missing values:
Series([], dtype: int64)

olist_orders_dataset.csv
Shape: (99441, 8

In [11]:
# ============================================================
# Q3 - DATA CLEANING + MERGING + FEATURE ENGINEERING
# ============================================================

# Make copies
orders = files["olist_orders_dataset.csv"].copy()
order_items = files["olist_order_items_dataset.csv"].copy()
payments = files["olist_order_payments_dataset.csv"].copy()
reviews = files["olist_order_reviews_dataset.csv"].copy()
customers = files["olist_customers_dataset.csv"].copy()
products = files["olist_products_dataset.csv"].copy()
sellers = files["olist_sellers_dataset.csv"].copy()
translation = files["product_category_name_translation.csv"].copy()

# ------------------------------------------------------------
# 1. Remove duplicate geolocation records
# ------------------------------------------------------------

geolocation = files["olist_geolocation_dataset.csv"].copy()
geolocation = geolocation.drop_duplicates()

print("Geolocation after duplicate removal:", geolocation.shape)


# ------------------------------------------------------------
# 2. Convert date columns to datetime
# ------------------------------------------------------------

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")


# ------------------------------------------------------------
# 3. Handle product missing values
# ------------------------------------------------------------

products["product_category_name"] = (
    products["product_category_name"]
    .fillna("unknown")
)

# Numeric product attributes are not required for our main
# business analysis, so missing values are retained as NaN
# rather than inventing measurements.


# ------------------------------------------------------------
# 4. Handle review data
# ------------------------------------------------------------

# Review text is optional for our main quantitative analysis.
# Keep missing text as empty strings.
reviews["review_comment_title"] = (
    reviews["review_comment_title"].fillna("")
)

reviews["review_comment_message"] = (
    reviews["review_comment_message"].fillna("")
)


# ------------------------------------------------------------
# 5. Translate product categories
# ------------------------------------------------------------

products = products.merge(
    translation,
    on="product_category_name",
    how="left"
)

products["product_category_english"] = (
    products["product_category_name_english"]
    .fillna(products["product_category_name"])
    .fillna("Unknown")
)


# ------------------------------------------------------------
# 6. Aggregate payments at ORDER level
#    Prevents one-to-many payment records from duplicating items
# ------------------------------------------------------------

payment_summary = (
    payments
    .groupby("order_id")
    .agg(
        payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max"),
        payment_methods=("payment_type", "nunique")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 7. Aggregate reviews at ORDER level
# ------------------------------------------------------------

review_summary = (
    reviews
    .groupby("order_id")
    .agg(
        review_score=("review_score", "mean"),
        review_count=("review_id", "nunique")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 8. Merge ORDER + CUSTOMER
# ------------------------------------------------------------

analysis = order_items.merge(
    orders,
    on="order_id",
    how="left"
)

analysis = analysis.merge(
    customers[
        [
            "customer_id",
            "customer_unique_id",
            "customer_zip_code_prefix",
            "customer_city",
            "customer_state"
        ]
    ],
    on="customer_id",
    how="left"
)


# ------------------------------------------------------------
# 9. Merge PRODUCT
# ------------------------------------------------------------

analysis = analysis.merge(
    products[
        [
            "product_id",
            "product_category_english",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ],
    on="product_id",
    how="left"
)


# ------------------------------------------------------------
# 10. Merge SELLER
# ------------------------------------------------------------

analysis = analysis.merge(
    sellers[
        [
            "seller_id",
            "seller_zip_code_prefix",
            "seller_city",
            "seller_state"
        ]
    ],
    on="seller_id",
    how="left"
)


# ------------------------------------------------------------
# 11. Merge PAYMENT SUMMARY
# ------------------------------------------------------------

analysis = analysis.merge(
    payment_summary,
    on="order_id",
    how="left"
)


# ------------------------------------------------------------
# 12. Merge REVIEW SUMMARY
# ------------------------------------------------------------

analysis = analysis.merge(
    review_summary,
    on="order_id",
    how="left"
)


# ------------------------------------------------------------
# 13. Create calculated fields
# ------------------------------------------------------------

analysis["item_total_value"] = (
    analysis["price"] + analysis["freight_value"]
)

analysis["delivery_days"] = (
    analysis["order_delivered_customer_date"]
    - analysis["order_purchase_timestamp"]
).dt.total_seconds() / 86400

analysis["estimated_delivery_days"] = (
    analysis["order_estimated_delivery_date"]
    - analysis["order_purchase_timestamp"]
).dt.total_seconds() / 86400

analysis["delivery_delay_days"] = (
    analysis["order_delivered_customer_date"]
    - analysis["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

analysis["purchase_month"] = (
    analysis["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)


# ------------------------------------------------------------
# 14. Categorize delivery performance
# ------------------------------------------------------------

analysis["delivery_performance"] = np.select(
    [
        analysis["delivery_delay_days"].isna(),
        analysis["delivery_delay_days"] <= 0,
        analysis["delivery_delay_days"] > 0
    ],
    [
        "Not Delivered / Missing Date",
        "On Time",
        "Delayed"
    ],
    default="Unknown"
)


# ------------------------------------------------------------
# 15. Remove exact duplicate rows created during processing
# ------------------------------------------------------------

analysis = analysis.drop_duplicates()


# ------------------------------------------------------------
# 16. Final overview
# ------------------------------------------------------------

print("FINAL ANALYSIS DATASET")
print("Rows:", analysis.shape[0])
print("Columns:", analysis.shape[1])

print("\nMissing values in key fields:")
print(
    analysis[
        [
            "order_id",
            "product_id",
            "seller_id",
            "price",
            "freight_value",
            "item_total_value",
            "review_score"
        ]
    ].isna().sum()
)

print("\nDelivery performance:")
print(analysis["delivery_performance"].value_counts(dropna=False))


# ------------------------------------------------------------
# 17. Save final processed dataset
# ------------------------------------------------------------

output_path = "/content/processed_olist_analysis_final.csv"

analysis.to_csv(
    output_path,
    index=False
)

print("\nSaved:", output_path)

Geolocation after duplicate removal: (738332, 5)
FINAL ANALYSIS DATASET
Rows: 112650
Columns: 37

Missing values in key fields:
order_id              0
product_id            0
seller_id             0
price                 0
freight_value         0
item_total_value      0
review_score        942
dtype: int64

Delivery performance:
delivery_performance
On Time                         101481
Delayed                           8715
Not Delivered / Missing Date      2454
Name: count, dtype: int64

Saved: /content/processed_olist_analysis_final.csv


In [12]:
# Q4 - Business Insights

# 1. Overall metrics
print("===== OVERALL =====")
print("Total Revenue:", round(analysis["item_total_value"].sum(), 2))
print("Total Orders:", analysis["order_id"].nunique())
print("Total Customers:", analysis["customer_unique_id"].nunique())
print("Total Sellers:", analysis["seller_id"].nunique())
print("Average Order Value:",
      round(analysis["item_total_value"].sum() /
            analysis["order_id"].nunique(), 2))


# 2. Top categories
print("\n===== TOP 10 CATEGORIES =====")

category_sales = (
    analysis.groupby("product_category_english")
    .agg(
        revenue=("item_total_value", "sum"),
        orders=("order_id", "nunique")
    )
    .sort_values("revenue", ascending=False)
)

print(category_sales.head(10))


# 3. Top sellers
print("\n===== TOP 10 SELLERS =====")

seller_sales = (
    analysis.groupby("seller_id")
    .agg(
        revenue=("item_total_value", "sum"),
        orders=("order_id", "nunique")
    )
    .sort_values("revenue", ascending=False)
)

print(seller_sales.head(10))


# 4. Monthly trend
print("\n===== MONTHLY TREND =====")

monthly_sales = (
    analysis.groupby("purchase_month")
    .agg(
        revenue=("item_total_value", "sum"),
        orders=("order_id", "nunique")
    )
    .reset_index()
)

print(monthly_sales)


# 5. Delivery performance
print("\n===== DELIVERY PERFORMANCE =====")

delivery = (
    analysis.groupby("delivery_performance")
    .agg(
        orders=("order_id", "nunique"),
        avg_delivery_days=("delivery_days", "mean"),
        avg_review_score=("review_score", "mean")
    )
)

print(delivery)


# 6. Review score
print("\n===== REVIEW SCORES =====")

print(
    analysis.groupby("review_score")["order_id"]
    .nunique()
    .sort_index()
)

===== OVERALL =====
Total Revenue: 15843553.24
Total Orders: 98666
Total Customers: 95420
Total Sellers: 3095
Average Order Value: 160.58

===== TOP 10 CATEGORIES =====
                             revenue  orders
product_category_english                    
health_beauty             1441248.07    8836
watches_gifts             1305541.61    5624
bed_bath_table            1241681.72    9417
sports_leisure            1156656.48    7720
computers_accessories     1059272.40    6689
furniture_decor            902511.79    6449
housewares                 778397.77    5884
cool_stuff                 719329.95    3632
auto                       685384.32    3897
garden_tools               584219.21    3518

===== TOP 10 SELLERS =====
                                    revenue  orders
seller_id                                          
4869f7a5dfa277a7dca6462dcf3b52b2  249640.70    1132
7c67e1448b00f6e969d365cea6b010ab  239536.44     982
53243585a1d6dc2643021fd1853d8905  235856.68     358
4a3